# Fast and Slow: Colab entry point

**Can a System One decision model (Kev-4B, an open JEV-style model) take over a planner's decidable calls, and how does it learn compared with classic RL?**
The spec, with its feature map and the reasoning behind each decision, is in `docs/spec.md`.

This notebook is deliberately thin: every cell calls one `fns` command from the tested `fast_and_slow` package.

**Built for a compute-unit budget (about 76 CU, spent in sessions of about 15).**
- Only Kev runs here, on the GPU. PPO needs only a CPU, so it runs on your laptop for free (section 8).
- `kev-check` measures Kev's real speed on whichever GPU you got, and `plan` prices every stage in compute units *before* you spend them.
- Kev RL stops at a checkpoint when the session's share runs out (`SESSION_CU`), and the next session carries on.

**Built for disconnects.** Colab Pro sessions end after about 24 hours at most, often sooner on an H100, and drop when idle.
- Finished stages are skipped.
- Kev RL resumes exactly from its last checkpoint (every 3 updates, saved to Google Drive).
- After a disconnect: run sections 1–3, then the cell you were in.

## 1. Settings

In [ ]:
REPO = "https://github.com/carlosdonado10/fast-and-slow.git"   # private repo: add a GITHUB_TOKEN secret (key icon, left sidebar)
BRANCH = "main"
PROFILE = "full"                                                # "smoke" runs everything in minutes with a fake Kev
RUNS = "/content/drive/MyDrive/fast-and-slow/runs"              # survives disconnects; your laptop reads it too
CU_PER_HOUR = 15        # your GPU's rate: Colab menu > Runtime > View resources. Check which GPU you got (40 GB = A100)
BUDGET_CU = 76          # what's left on your account
SESSION_CU = 15         # what you'll spend per sitting; every session first pays ~2 CU of setup (model download)
MAX_MINUTES = int(SESSION_CU / CU_PER_HOUR * 60) - 15   # Kev RL pauses at a checkpoint after this; the next session continues

## 2. Setup: Drive, code, environment

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RUNS, exist_ok=True)

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = REPO.replace("https://", f"https://{token}@") if token else REPO

if not os.path.exists("/content/fast-and-slow"):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", url, "/content/fast-and-slow"], check=True)
else:
    subprocess.run(["git", "-C", "/content/fast-and-slow", "pull", "--ff-only"], check=True)
os.chdir("/content/fast-and-slow/backend")

In [ ]:
%%bash
cd /content/fast-and-slow/backend
command -v uv || curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"
uv sync --extra gpu --frozen 2>&1 | tail -3
nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import time
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["PYTHONUNBUFFERED"] = "1"   # stream progress lines as they happen, not in bursts
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less memory lost to fragmentation in RL
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # optional secret: faster model downloads
except Exception:
    pass

def fns(*args):
    """Run one study stage and stream its output. Long stages print progress with an estimate of the time left."""
    cmd = ["uv", "run", "--frozen", "fns", *map(str, args), "--profile", PROFILE, "--runs", RUNS]
    started = time.monotonic()
    print(f"$ fns {' '.join(map(str, args))}", flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    m, s = divmod(round(time.monotonic() - started), 60)
    if p.wait() != 0:
        raise RuntimeError(f"fns {args[0]} failed after {m}m{s:02d}s")
    print(f"fns {args[0]} finished in {m}m{s:02d}s\n", flush=True)

fns("status")

## 3. Preflight and the bill
`kev-check` loads Kev-4B, confirms gradients reach the adapter, and times real requests on this GPU (about a minute).
`plan` then prices every stage and says how many sessions of `SESSION_CU` it takes. **If it says over budget, follow its suggestion before going further.**

In [ ]:
fns("notes")       # Phi-4-mini writes the inbox notes once (about 10 min); kev-check uses them
fns("kev-check")
fns("plan", "--cu-per-hour", CU_PER_HOUR, "--budget", BUDGET_CU, "--session-cu", SESSION_CU)

## 4. References (CPU, about a minute)

In [ ]:
fns("refs")

## 5. Kev zero-shot and in-context (no training)

In [ ]:
fns("kev")

## 6. Kev + RL
Each run continues from its checkpoint. All runs together share `MAX_MINUTES` per session, then pause at a checkpoint.
Every update prints a line with how far along the run is, roughly how long is left and when it pauses. Run this cell again (in this session or the next)
until every run says *finished*. Check progress with `fns("status")`.

In [ ]:
import yaml, pathlib
cfg = yaml.safe_load(pathlib.Path(RUNS, PROFILE, "config.json").read_text())
todo = [(h, seed) for seed in cfg["seeds"] for h in cfg["kev"]["rl_horizons"]]
session_end = time.monotonic() + MAX_MINUTES * 60
for i, (h, seed) in enumerate(todo, 1):
    left = (session_end - time.monotonic()) / 60
    if left < 2:
        print("This session's RL time is used up. Run this cell again in the next session to continue.")
        break
    print(f"=== RL run {i} of {len(todo)}: horizon {h}, seed {seed} ({left:.0f} min of this session's RL time left) ===")
    fns("kev-rl", "--horizon", h, "--seed", seed, "--max-minutes", int(left))

In [ ]:
fns("status")

## 6b. Regime changes (H6): Kev's part, a few minutes
Kev on each new world (told about the change, and not told) and the weekly "has something changed?" alarm over
simulated years. The PPO retraining half runs on your laptop (section 8).

In [ ]:
fns("shift", "--part", "gpu")

**When everything here is done, disconnect the runtime** (Runtime > Disconnect and delete runtime) so it stops using units.

## 7. Export the report
Run this after PPO (section 8) has finished on your laptop, or now for a partial report.

In [ ]:
report = f"{RUNS}/report.json"
fns("export", "--out", report)
from google.colab import files
files.download(report)   # put it at frontend/public/data/report.json

## 8. PPO on your laptop (free)
PPO is CPU-only, so it doesn't need Colab. With [Google Drive for desktop](https://www.google.com/drive/download/) syncing the same `runs` folder:

```bash
cd fast-and-slow/backend
uv sync --extra gpu        # the `emb` variant needs the sentence encoder
RUNS="<your Drive folder>/fast-and-slow/runs"
uv run fns ppo-all --profile full --runs "$RUNS"
uv run fns shift --part cpu --profile full --runs "$RUNS"   # H6: retrain the deployed PPO on each new world
```

It needs the note bank from section 3. Like everything else, it resumes after interruptions (every 25k decisions).